In [1]:
from transformers import pipeline

def execute_pipeline_inference():
    """
    Initializes a text classification pipeline using
    a pre-trained DistilBERT model.
    """

    # Load the sentiment analysis pipeline
    sentiment_analyzer = pipeline(
        task="sentiment-analysis",
        model="distilbert-base-uncased-finetuned-sst-2-english",
        device=-1      # Use CPU (0 for GPU)
    )

    # Sample input texts
    evaluation_texts = [
        "The instructional presentation was incredibly clear and offered profound, intuitive explanations.",
        "The network architecture is painfully slow, poorly optimized, and completely lacks clear documentation."
    ]

    # Perform sentiment analysis
    inference_outputs = sentiment_analyzer(evaluation_texts)

    # Display results
    for text, output in zip(evaluation_texts, inference_outputs):
        print(f"Observed Text: '{text}'")
        print(f"Calculated Label: {output['label']} | Confidence Score: {output['score']:.4f}\n")


if __name__ == "__main__":
    execute_pipeline_inference()

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Observed Text: 'The instructional presentation was incredibly clear and offered profound, intuitive explanations.'
Calculated Label: POSITIVE | Confidence Score: 0.9996

Observed Text: 'The network architecture is painfully slow, poorly optimized, and completely lacks clear documentation.'
Calculated Label: NEGATIVE | Confidence Score: 0.9998



In [6]:
import torch
from transformers import AutoTokenizer

def generate_tensor_inputs():
    """
    Demonstrates manual vocabulary mapping, text truncation,
    and padding rules using Hugging Face.
    """

    model_identifier = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_identifier)

    sample_phrases = [
        "Transformers bypass sequential bottlenecks.",
        "Attention mechanisms preserve contextual semantics perfectly."
    ]

    encoded_inputs = tokenizer(
        sample_phrases,
        padding=True,
        truncation=True,
        max_length=16,
        return_tensors="pt"
    )

    print("Generated Input Token Indices (input_ids):")
    print(encoded_inputs["input_ids"])

    print("\nGenerated Parallel Attention Masks:")
    print(encoded_inputs["attention_mask"])

    return encoded_inputs


if __name__ == "__main__":
    tensor_batch = generate_tensor_inputs()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Generated Input Token Indices (input_ids):
tensor([[  101, 19081, 11826, 25582,  5835, 18278,  2015,  1012,   102,     0],
        [  101,  3086, 10595,  7969,  6123,  8787, 28081,  6669,  1012,   102]])

Generated Parallel Attention Masks:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [8]:
import torch
from transformers import AutoTokenizer, AutoModel

def extract_hidden_state_embeddings(text_batch):
    """
    Extracts high-dimensional latent vectors directly from the final encoder layer
    of a pre-trained Transformer backbone.
    """

    model_name = "distilbert-base-uncased"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    # Pre-process raw inputs into PyTorch tensors
    inputs = tokenizer(
        text_batch,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    # Disable gradients during inference
    model.eval()

    with torch.no_grad():
        model_outputs = model(**inputs)

    # Output shape:
    # [Batch Size, Sequence Length, Hidden Dimension]
    last_hidden_states = model_outputs.last_hidden_state

    # Extract the first token embedding (CLS-equivalent for DistilBERT)
    sentence_level_embeddings = last_hidden_states[:, 0, :].numpy()

    print("Full Encoder Output Matrix Shape:", last_hidden_states.shape)
    print("Extracted Sentence Context Vector Shape:", sentence_level_embeddings.shape)

    return sentence_level_embeddings


if __name__ == "__main__":

    sample_corpus = [
        "Highly scalable context vectors.",
        "Extracting mathematical features."
    ]

    embeddings = extract_hidden_state_embeddings(sample_corpus)

    print("\nSentence Embeddings:")
    print(embeddings)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Full Encoder Output Matrix Shape: torch.Size([2, 8, 768])
Extracted Sentence Context Vector Shape: (2, 768)

Sentence Embeddings:
[[-0.5616026  -0.20605405 -0.2804925  ... -0.19543813 -0.34261146
   0.45600554]
 [-0.67307174 -0.28650787 -0.5085631  ... -0.09904305  0.17002013
   0.39435494]]


In [10]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


def execute_custom_classifier_training():
    """
    Combines frozen high-dimensional Transformer context embeddings
    with a distinct downstream classifier head to create a hybrid model.
    """

    print("Simulating upstream extraction step...")

    # Generate mock transformer embeddings
    np.random.seed(42)
    mock_transformer_embeddings = np.random.randn(100, 768)

    # Generate binary labels
    mock_labels = np.random.randint(0, 2, size=(100,))

    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        mock_transformer_embeddings,
        mock_labels,
        test_size=0.2,
        random_state=42
    )

    # Train Logistic Regression classifier
    custom_head = LogisticRegression(
        C=1.0,
        max_iter=1000,
        solver='lbfgs'
    )

    custom_head.fit(X_train, y_train)

    # Predict on test data
    predictions = custom_head.predict(X_test)

    # Predict probabilities
    prediction_probabilities = custom_head.predict_proba(X_test)[:, 1]

    print(f"Custom classifier trained on {X_train.shape[0]} samples.")
    print(f"Prediction array shape: {predictions.shape}")

    return y_test, predictions, prediction_probabilities


if __name__ == "__main__":
    y_true, y_pred, y_prob = execute_custom_classifier_training()

Simulating upstream extraction step...
Custom classifier trained on 80 samples.
Prediction array shape: (20,)


In [11]:
import numpy as np


def calculate_professional_metrics(true_labels, predicted_labels):
    """
    Computes Precision, Recall, F1-Score, and Accuracy
    from true and predicted labels.
    """

    # Convert to NumPy arrays
    y_true = np.array(true_labels)
    y_pred = np.array(predicted_labels)

    # Calculate confusion matrix values
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))

    # Calculate metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    f1_score = (
        2 * (precision * recall) / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    accuracy = (tp + tn) / len(y_true)

    print("========== MACHINE LEARNING REPORT ==========")
    print(f"True Positives : {tp}")
    print(f"False Positives: {fp}")
    print(f"True Negatives : {tn}")
    print(f"False Negatives: {fn}")
    print("---------------------------------------------")
    print(f"Precision : {precision * 100:.2f}%")
    print(f"Recall    : {recall * 100:.2f}%")
    print(f"F1-Score  : {f1_score * 100:.2f}%")
    print(f"Accuracy  : {accuracy * 100:.2f}%")
    print("=============================================")

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1_score,
        "accuracy": accuracy
    }


if __name__ == "__main__":

    target_ground_truth = [1, 0, 1, 1, 0, 0, 1, 0, 1, 1]

    model_predictions = [1, 0, 0, 1, 0, 1, 1, 0, 1, 0]

    metrics = calculate_professional_metrics(
        target_ground_truth,
        model_predictions
    )

========== MACHINE LEARNING REPORT ==========
True Positives : 4
False Positives: 1
True Negatives : 3
False Negatives: 2
---------------------------------------------
Precision : 80.00%
Recall    : 66.67%
F1-Score  : 72.73%
Accuracy  : 70.00%
